# SpyCloud Graph Investigation Blueprint
## Identity Exposure -> Attack Path Analysis with Sentinel Graph

This notebook combines SpyCloud dark web intelligence with Microsoft Sentinel Graph
for comprehensive attack path analysis.

**Prerequisites:**
- Microsoft Sentinel workspace with SpyCloud data connector
- Sentinel Graph enabled (GA since Dec 2025)
- Python packages: `pip install -r notebooks/requirements.txt`
- Azure credentials configured

**Sections:**
1. Setup & Sentinel Connection
2. SpyCloud Exposure Landscape
3. Identity-Asset Graph Construction
4. Blast Radius Analysis
5. AI-Powered Analysis
6. Export & Reporting

In [ ]:
# Setup & Configuration
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import matplotlib.pyplot as plt
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import networkx as nx
from datetime import datetime, timedelta
from IPython.display import display, HTML, Markdown
import json
import os

# MSTICPy setup
import msticpy as mp
mp.init_notebook(namespace=globals(), verbosity=0)

print('Libraries loaded successfully.')
print(f'MSTICPy version: {mp.__version__}')
print(f'Investigation started: {datetime.utcnow().isoformat()}Z')

In [ ]:
# Connect to Microsoft Sentinel
qry_prov = QueryProvider('MSSentinel')

# Interactive login - opens browser for Azure AD authentication
# Replace YOUR_TENANT_ID and YOUR_WORKSPACE_ID with actual values
qry_prov.connect(
    connection_str="loganalytics://code().tenant('YOUR_TENANT_ID')"
    ".workspace('YOUR_WORKSPACE_ID')"
)
print('Connected to Sentinel workspace.')

In [ ]:
# Configure AI provider (optional)
# Option 1: OpenAI
# os.environ['OPENAI_API_KEY'] = 'sk-...'  # Or load from .env

# Option 2: Azure OpenAI
# os.environ['AZURE_OPENAI_ENDPOINT'] = 'https://your-resource.openai.azure.com/'
# os.environ['AZURE_OPENAI_KEY'] = 'your-key'
# os.environ['AZURE_OPENAI_DEPLOYMENT'] = 'gpt-4o'

AI_AVAILABLE = bool(os.environ.get('OPENAI_API_KEY') or os.environ.get('AZURE_OPENAI_KEY'))
print(f"AI Analysis: {'Available' if AI_AVAILABLE else 'Not configured (optional)'}")

## Section 1: SpyCloud Exposure Landscape
Query the SpyCloud breach watchlist to understand the organization's current exposure posture.

In [ ]:
# Organization-wide exposure overview
exposure_overview = qry_prov.exec_query("""
SpyCloudBreachWatchlist_CL
| where TimeGenerated >= ago(90d)
| summarize
    TotalExposures = count(),
    CriticalExposures = countif(severity_d >= 20),
    PlaintextCredentials = countif(isnotempty(password_plaintext_s)),
    UniqueUsers = dcount(email_s),
    UniqueDomains = dcount(domain_s),
    UniqueDevices = dcount(infected_machine_id_s),
    UniqueBreachSources = dcount(source_id_d)
""")

if exposure_overview is not None and not exposure_overview.empty:
    eo = exposure_overview.iloc[0]
    display(Markdown('### Organization Exposure Summary (90 Days)'))
    summary = f"""
| Metric | Value |
|--------|-------|
| Total Exposures | **{eo.get('TotalExposures', 0):,}** |
| Critical (Sev >= 20) | **{eo.get('CriticalExposures', 0):,}** |
| Plaintext Passwords | **{eo.get('PlaintextCredentials', 0):,}** |
| Unique Users | **{eo.get('UniqueUsers', 0):,}** |
| Infected Devices | **{eo.get('UniqueDevices', 0):,}** |
"""
    display(Markdown(summary))
else:
    print('No SpyCloud data found. Verify the data connector is active.')

In [ ]:
# Severity distribution over time
severity_trend = qry_prov.exec_query("""
SpyCloudBreachWatchlist_CL
| where TimeGenerated >= ago(90d)
| summarize
    Sev2 = countif(severity_d == 2),
    Sev5 = countif(severity_d == 5),
    Sev20 = countif(severity_d == 20),
    Sev25 = countif(severity_d == 25)
    by bin(TimeGenerated, 1d)
| order by TimeGenerated asc
""")

if severity_trend is not None and not severity_trend.empty:
    fig = go.Figure()
    fig.add_trace(go.Scatter(x=severity_trend['TimeGenerated'], y=severity_trend['Sev25'],
                             name='Sev 25 (Critical)', fill='tozeroy', line=dict(color='#dc3545')))
    fig.add_trace(go.Scatter(x=severity_trend['TimeGenerated'], y=severity_trend['Sev20'],
                             name='Sev 20 (High)', fill='tozeroy', line=dict(color='#fd7e14')))
    fig.add_trace(go.Scatter(x=severity_trend['TimeGenerated'], y=severity_trend['Sev5'],
                             name='Sev 5 (Standard)', fill='tozeroy', line=dict(color='#ffc107')))
    fig.add_trace(go.Scatter(x=severity_trend['TimeGenerated'], y=severity_trend['Sev2'],
                             name='Sev 2 (Low)', fill='tozeroy', line=dict(color='#6c757d')))
    fig.update_layout(title='Exposure Severity Trend (90 Days)',
                      xaxis_title='Date', yaxis_title='Exposures',
                      height=450, template='plotly_dark')
    fig.show()
else:
    print('No trend data available.')

## Section 2: Identity-Asset Graph Construction
Build a graph connecting compromised identities to assets they access.

In [ ]:
# Get exposed users with breach context
exposed_users = qry_prov.exec_query("""
SpyCloudBreachWatchlist_CL
| where TimeGenerated >= ago(30d)
| where severity_d >= 20
| summarize
    ExposureCount = count(),
    MaxSeverity = max(severity_d),
    HasPlaintext = countif(isnotempty(password_plaintext_s)) > 0,
    InfectedDevices = make_set(infected_machine_id_s),
    TargetDomains = make_set(target_domain_s)
    by email_s
| order by MaxSeverity desc, ExposureCount desc
| take 50
""")

signin_data = None
if exposed_users is not None and not exposed_users.empty:
    user_list = "','".join(exposed_users['email_s'].tolist())
    signin_data = qry_prov.exec_query(f"""
    SigninLogs
    | where TimeGenerated >= ago(30d)
    | where UserPrincipalName in~ ('{user_list}')
    | summarize
        SigninCount = count(),
        UniqueApps = make_set(AppDisplayName),
        UniqueIPs = make_set(IPAddress)
        by UserPrincipalName
    """)
    display(Markdown(f'### Found {len(exposed_users)} users with critical exposures'))
    display(exposed_users.head(20))
else:
    print('No critical exposures found in the last 30 days.')

In [ ]:
# Build NetworkX graph
G = nx.DiGraph()

if exposed_users is not None:
    for _, row in exposed_users.iterrows():
        email = row['email_s']
        sev = row.get('MaxSeverity', 0)
        color = '#dc3545' if sev >= 25 else '#fd7e14' if sev >= 20 else '#ffc107'
        G.add_node(email, type='user', severity=sev, color=color,
                   exposures=row.get('ExposureCount', 0))

        devices = row.get('InfectedDevices', [])
        if isinstance(devices, str):
            try: devices = json.loads(devices)
            except: devices = []
        for dev in devices:
            if dev:
                G.add_node(dev, type='device', color='#17a2b8')
                G.add_edge(email, dev, relationship='infected_on')

        domains = row.get('TargetDomains', [])
        if isinstance(domains, str):
            try: domains = json.loads(domains)
            except: domains = []
        for dom in domains:
            if dom:
                G.add_node(dom, type='application', color='#6f42c1')
                G.add_edge(email, dom, relationship='credential_for')

if signin_data is not None:
    for _, row in signin_data.iterrows():
        upn = row['UserPrincipalName']
        apps = row.get('UniqueApps', [])
        if isinstance(apps, str):
            try: apps = json.loads(apps)
            except: apps = []
        for app in apps:
            if app and app != 'unknown':
                G.add_node(app, type='application', color='#6f42c1')
                G.add_edge(upn, app, relationship='signed_into')

uc = sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'user')
dc = sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'device')
ac = sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'application')
display(Markdown(f'### Identity-Asset Graph\n- **Nodes:** {G.number_of_nodes()} ({uc} users, {dc} devices, {ac} apps)\n- **Edges:** {G.number_of_edges()} relationships'))

In [ ]:
# Visualize the identity-asset graph
pos = nx.spring_layout(G, k=2, iterations=50, seed=42)

edge_x, edge_y = [], []
for edge in G.edges():
    x0, y0 = pos[edge[0]]
    x1, y1 = pos[edge[1]]
    edge_x.extend([x0, x1, None])
    edge_y.extend([y0, y1, None])

edge_trace = go.Scatter(x=edge_x, y=edge_y, mode='lines',
                        line=dict(width=0.5, color='#888'), hoverinfo='none')

node_traces = []
for node_type, shape, default_color in [
    ('user', 'circle', '#dc3545'),
    ('device', 'square', '#17a2b8'),
    ('application', 'diamond', '#6f42c1')]:
    nodes = [(n, d) for n, d in G.nodes(data=True) if d.get('type') == node_type]
    if not nodes: continue
    xs = [pos[n][0] for n, _ in nodes]
    ys = [pos[n][1] for n, _ in nodes]
    colors = [d.get('color', default_color) for _, d in nodes]
    labels = [n for n, _ in nodes]
    trace = go.Scatter(x=xs, y=ys, mode='markers+text',
        marker=dict(size=15 if node_type == 'user' else 10, color=colors,
                    symbol=shape, line=dict(width=1, color='white')),
        text=[n[:20] for n in labels], textposition='top center',
        textfont=dict(size=8), name=node_type.capitalize(), hovertext=labels)
    node_traces.append(trace)

fig = go.Figure(data=[edge_trace] + node_traces)
fig.update_layout(title='SpyCloud Identity-Asset Relationship Graph',
    showlegend=True, hovermode='closest',
    xaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    yaxis=dict(showgrid=False, zeroline=False, showticklabels=False),
    height=700, template='plotly_dark')
fig.show()

## Section 3: Blast Radius Analysis
Calculate the blast radius for each critically exposed user.

In [ ]:
# Calculate blast radius
blast_radius_results = []

if exposed_users is not None:
    for _, row in exposed_users.head(10).iterrows():
        email = row['email_s']
        if email not in G: continue
        reachable = nx.descendants(G, email)
        by_type = {'device': 0, 'application': 0, 'user': 0}
        for node in reachable:
            ntype = G.nodes[node].get('type', 'unknown')
            if ntype in by_type: by_type[ntype] += 1
        blast_radius_results.append({
            'User': email, 'Severity': row.get('MaxSeverity', 0),
            'Exposures': row.get('ExposureCount', 0),
            'Reachable Devices': by_type['device'],
            'Reachable Apps': by_type['application'],
            'Total Blast Radius': len(reachable)})

if blast_radius_results:
    br_df = pd.DataFrame(blast_radius_results)
    display(Markdown('### Blast Radius by User'))
    display(br_df)
    fig = px.bar(br_df, x='User', y='Total Blast Radius',
                 color='Severity', color_continuous_scale='RdYlGn_r',
                 title='Blast Radius by Compromised User',
                 hover_data=['Reachable Devices', 'Reachable Apps'])
    fig.update_layout(height=450, xaxis_tickangle=-45, template='plotly_dark')
    fig.show()
else:
    print('No blast radius data to display.')

## Section 4: AI-Powered Analysis (Optional)
Generate an AI-powered investigation summary. Requires `OPENAI_API_KEY` or `AZURE_OPENAI_KEY`.

In [ ]:
# AI-powered analysis
if AI_AVAILABLE and exposed_users is not None and not exposed_users.empty:
    try:
        import openai
        if os.environ.get('AZURE_OPENAI_KEY'):
            client = openai.AzureOpenAI(
                azure_endpoint=os.environ['AZURE_OPENAI_ENDPOINT'],
                api_key=os.environ['AZURE_OPENAI_KEY'],
                api_version='2024-08-01-preview')
            model = os.environ.get('AZURE_OPENAI_DEPLOYMENT', 'gpt-4o')
        else:
            client = openai.OpenAI(api_key=os.environ['OPENAI_API_KEY'])
            model = 'gpt-4o'

        context = {
            'exposure_summary': exposure_overview.to_dict() if exposure_overview is not None else {},
            'top_users': exposed_users.head(10).to_dict() if exposed_users is not None else {},
            'blast_radius': blast_radius_results[:10],
            'graph_stats': {'nodes': G.number_of_nodes(), 'edges': G.number_of_edges()}}

        response = client.chat.completions.create(
            model=model,
            messages=[
                {'role': 'system', 'content': 'You are SCORCH, SpyCloud expert identity threat analyst. Provide detailed analysis with MITRE ATT&CK mapping.'},
                {'role': 'user', 'content': f'Analyze dark web exposure data and provide investigation summary with remediation plan:\n\n{json.dumps(context, indent=2, default=str)}'}],
            temperature=0.3, max_tokens=3000)

        display(Markdown('### AI Investigation Summary'))
        display(Markdown(response.choices[0].message.content))
    except Exception as e:
        print(f'AI analysis error: {e}')
else:
    if not AI_AVAILABLE:
        display(Markdown('*AI analysis skipped. Set OPENAI_API_KEY or AZURE_OPENAI_KEY to enable.*'))

## Section 5: Export & Reporting
Export findings for incident documentation or executive reporting.

In [ ]:
# Export investigation results
report_data = {
    'investigation_timestamp': datetime.utcnow().isoformat() + 'Z',
    'exposure_overview': exposure_overview.to_dict() if exposure_overview is not None and not exposure_overview.empty else {},
    'critical_users': exposed_users.to_dict() if exposed_users is not None and not exposed_users.empty else {},
    'blast_radius': blast_radius_results,
    'graph_stats': {
        'total_nodes': G.number_of_nodes(),
        'total_edges': G.number_of_edges(),
        'user_nodes': sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'user'),
        'device_nodes': sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'device'),
        'app_nodes': sum(1 for _, d in G.nodes(data=True) if d.get('type') == 'application')}}

with open('investigation_report.json', 'w') as f:
    json.dump(report_data, f, indent=2, default=str)

display(Markdown('Investigation report saved to `investigation_report.json`'))
display(Markdown('Use this data for Sentinel incident comments, AI Engine executive reports, or compliance documentation.'))